In [64]:
import pandas as pd
import numpy as np

In [65]:
df = pd.read_csv('../data/raw/aggregated/crypto_data.csv')
df.sample(5)

,date,open,high,low,close,volume,token
75470,2021-01-31 22:00:00,0.34170,0.34697,0.34137,0.34602,3099098,ADA
670711,2023-03-19 08:00:00,5.99300,6.05700,5.94500,5.98100,1533563,FIL
1080389,2019-04-11 00:00:00,70.64000,71.17000,69.72000,69.72000,14422,XMR
481887,2021-06-24 17:00:00,0.24411,0.24746,0.24224,0.24561,38188562,DOGE
268748,2018-04-03 04:00:00,12.52910,12.59730,12.33000,12.43000,2543361,BNB


In [66]:
df['date'] = pd.to_datetime(df['date'])

## unused tokens

In [67]:
# For feature engineering; we do not intend to trade these 
# symbols because the market is too efficient.
btc = df[df['token'] == 'BTC']
eth = df[df['token'] == 'ETH']

to_drop = ['open', 'high', 'low', 'volume', 'token']
btc = btc.drop(columns= to_drop).rename(columns={'close': 'btc_close'})
eth = eth.drop(columns= to_drop).rename(columns={'close': 'eth_close'})

btc.info()

<class 'pandas.core.frame.DataFrame'>
Index: 54116 entries, 317303 to 371418
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   date       54116 non-null  datetime64[ns]
 1   btc_close  54116 non-null  float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 1.2 MB


In [68]:
print(len(df['token'].unique()))
df = df[~df['token'].isin({'ETH', 'BTC'})]
print(len(df['token'].unique()))

32
30


## next_close

In [69]:
# Create the target feature
df['next_close'] = df.groupby('token')['close'].shift(-1)
df.head()

,date,open,high,low,close,volume,token,next_close
0,2020-12-25 05:00:00,0.2000,3.0885,0.2000,2.5826,35530516,1INCH,2.5059
1,2020-12-25 06:00:00,2.5824,2.6900,2.2249,2.5059,22440875,1INCH,2.6237
2,2020-12-25 07:00:00,2.5152,2.8870,2.3609,2.6237,21300426,1INCH,2.6134
3,2020-12-25 08:00:00,2.6318,2.8247,2.4650,2.6134,17491813,1INCH,2.6365
4,2020-12-25 09:00:00,2.6104,2.7498,2.5629,2.6365,9919400,1INCH,2.7667


In [70]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1011829 entries, 0 to 1120060
Data columns (total 8 columns):
 #   Column      Non-Null Count    Dtype         
---  ------      --------------    -----         
 0   date        1011829 non-null  datetime64[ns]
 1   open        1011829 non-null  float64       
 2   high        1011829 non-null  float64       
 3   low         1011829 non-null  float64       
 4   close       1011829 non-null  float64       
 5   volume      1011829 non-null  int64         
 6   token       1011829 non-null  object        
 7   next_close  1011799 non-null  float64       
dtypes: datetime64[ns](1), float64(5), int64(1), object(1)
memory usage: 69.5+ MB


## OHLCV_log (log returns)

In [71]:
log_returns = lambda s: np.log(s / s.shift(1))  # noqa: E731
df['open_log'] = df.groupby('token')['open'].transform(log_returns)
df['high_log'] = df.groupby('token')['high'].transform(log_returns)
df['low_log'] = df.groupby('token')['low'].transform(log_returns)
df['close_log'] = df.groupby('token')['close'].transform(log_returns)
df['next_close_log'] = df.groupby('token')['next_close'].transform(log_returns)
df['volume_log'] = df.groupby('token')['volume'].transform(lambda x: np.log(1 + x))

In [72]:
df.head()

,date,open,high,low,close,volume,token,next_close,open_log,high_log,low_log,close_log,next_close_log,volume_log
0,2020-12-25 05:00:00,0.2000,3.0885,0.2000,2.5826,35530516,1INCH,2.5059,NaN,NaN,NaN,NaN,NaN,17.385903
1,2020-12-25 06:00:00,2.5824,2.6900,2.2249,2.5059,22440875,1INCH,2.6237,2.558157,-0.138144,2.409150,-0.030149,0.045938,16.926395
2,2020-12-25 07:00:00,2.5152,2.8870,2.3609,2.6237,21300426,1INCH,2.6134,-0.026367,0.070677,0.059331,0.045938,-0.003933,16.874238
3,2020-12-25 08:00:00,2.6318,2.8247,2.4650,2.6134,17491813,1INCH,2.6365,0.045316,-0.021816,0.043149,-0.003933,0.008800,16.677244
4,2020-12-25 09:00:00,2.6104,2.7498,2.5629,2.6365,9919400,1INCH,2.7667,-0.008165,-0.026874,0.038948,0.008800,0.048203,16.110003


## year + add eth & btc

In [73]:
df['year'] = df['date'].dt.year

In [74]:
df = df.merge(btc, on='date', how='inner')
df = df.merge(eth, on='date', how='inner')

In [75]:
df['eth_btc_ratio'] = df['eth_close'] / df['btc_close']

In [76]:
df['btc_close_log'] = df.groupby('token')['btc_close'].transform(log_returns)
df['eth_close_log'] = df.groupby('token')['eth_close'].transform(log_returns)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1011829 entries, 0 to 1011828
Data columns (total 20 columns):
 #   Column          Non-Null Count    Dtype         
---  ------          --------------    -----         
 0   date            1011829 non-null  datetime64[ns]
 1   open            1011829 non-null  float64       
 2   high            1011829 non-null  float64       
 3   low             1011829 non-null  float64       
 4   close           1011829 non-null  float64       
 5   volume          1011829 non-null  int64         
 6   token           1011829 non-null  object        
 7   next_close      1011799 non-null  float64       
 8   open_log        1011799 non-null  float64       
 9   high_log        1011799 non-null  float64       
 10  low_log         1011799 non-null  float64       
 11  close_log       1011799 non-null  float64       
 12  next_close_log  1011769 non-null  float64       
 13  volume_log      1011829 non-null  float64       
 14  year            10

## drop extracted columns & null-containing rows

## time_idx

In [78]:
# df['time_idx'] = df.groupby('token').cumcount()
df.head()

,date,open,high,low,close,volume,token,next_close,open_log,high_log,low_log,close_log,next_close_log,volume_log,year,btc_close,eth_close,eth_btc_ratio,btc_close_log,eth_close_log
0,2020-12-25 05:00:00,0.2000,3.0885,0.2000,2.5826,35530516,1INCH,2.5059,NaN,NaN,NaN,NaN,NaN,17.385903,2020,23599.22,609.26,0.025817,NaN,NaN
1,2020-12-25 06:00:00,2.5824,2.6900,2.2249,2.5059,22440875,1INCH,2.6237,2.558157,-0.138144,2.409150,-0.030149,0.045938,16.926395,2020,23625.46,613.93,0.025986,0.001111,0.007636
2,2020-12-25 07:00:00,2.5152,2.8870,2.3609,2.6237,21300426,1INCH,2.6134,-0.026367,0.070677,0.059331,0.045938,-0.003933,16.874238,2020,23649.82,612.92,0.025916,0.001031,-0.001646
3,2020-12-25 08:00:00,2.6318,2.8247,2.4650,2.6134,17491813,1INCH,2.6365,0.045316,-0.021816,0.043149,-0.003933,0.008800,16.677244,2020,23677.44,617.01,0.026059,0.001167,0.006651
4,2020-12-25 09:00:00,2.6104,2.7498,2.5629,2.6365,9919400,1INCH,2.7667,-0.008165,-0.026874,0.038948,0.008800,0.048203,16.110003,2020,23934.52,623.46,0.026049,0.010799,0.010399


# Cyclical features

## hour

## weekday

## month

## norm_day